<a href="https://colab.research.google.com/github/Om-merkle/Embedding_FT/blob/main/EMBEDDING_FT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers torch

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
text = "Embedding fine-tuning improves retireval quaility in RAG Systems."

In [ ]:
embedding = model.encode(text)

In [ ]:
embedding

In [ ]:
import numpy as np
print("Embedding Shape:", embedding.shape)
print("First 10 Values:", np.round(embedding[:10], 6))

In [ ]:
import numpy as np
def cosine_sim(a,b):
  return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)))

In [ ]:
documents = [
    "Embedding fine tuning imporoves retrieval in RAG",
    "LoRA reduces GPU memory usage",
    "Vector databases store embeddings."
]

query = "How to improve reterieval in RAG"

In [ ]:
doc_embs = model.encode(documents, convert_to_numpy=True)
q_emb = model.encode(query,convert_to_numpy=True)

In [ ]:
scores = [cosine_sim(q_emb,d_emb) for d_emb in doc_embs]
scores

In [ ]:
# print all scores

for doc, score in zip(documents,scores):
  print(f"Score: {score:.4}  |  {doc}")

In [ ]:
pairs = [
    ("Embedding improves Semantic search",
     "Fine-tuned embedding enhance retrieval"),

    ("Embedding improves semantic search",
     "I Bought a new car yesterday"),
]

In [ ]:
for s1, s2 in pairs:
  emb1, emb2 = model.encode([s1,s2], normalize_embeddings=True)
  score = cosine_sim(emb1,emb2)
  print(f"\nSentence 1 : {s1}")
  print(f"Sentence 2 : {s2}")
  print("Cosine Similarity:", round(score, 4))

# Keyword Search vs Vector Search

In [ ]:
import re

documents = [
    "Embedding fine tuning improves retrieval in RAG",
    "LoRA reduces GPU memory usage",
    "My car needs fuel.",
    "Vector databases store embeddings",
]


query = "How to improve retrieval in RAG?"


# Tokenize function

def tokenize(text):
  return re.findall(r"[a-z0-9]+", text.lower())

query_tokens = tokenize(query)

print("Query tokens :", query_tokens)
print("\n=== Keyword Search Scores ===")


for doc in documents:
  doc_tokens = tokenize(doc)

  # Count matching words

  matches = set(query_tokens) & set(doc_tokens)
  score = len(matches)

  print(f"Scores: {scores}  | Matched: {matches} |  {doc}")


In [ ]:
!pip install rank_bm25

In [ ]:
import re
from rank_bm25 import BM25Okapi

In [ ]:
import re

documents = [
    "Embedding fine tuning improves retrieval in RAG",
    "LoRA reduces GPU memory usage",
    "My car needs fuel.",
    "Vector databases store embeddings",
]


query = "How to improve retrieval in RAG?"


# Tokenize function

def tokenize(text):
  return re.findall(r"[a-z0-9]+", text.lower())

tokenized_corpus = [tokenize(d) for d in documents]
tokenized_query = tokenize(query)

bm25 = BM25Okapi(tokenized_corpus)

scores = bm25.get_scores(tokenized_query)


print("=== BM25 Scores (all docs) ===")
for doc, score in zip(documents, scores):
  print(f"{score:.4f}  |  {doc}")

In [ ]:
!pip install mteb

MTEB - (Massive Text Embedding Benchmark)


STSBenchmark = Semantic Textual Similarity Benchmark

In [ ]:
import mteb
from typing import Dict, Any, Optional

def eval_stsbenchmark(
    model_name: str,
    batch_size: int = 32,
    normalize_embeddings:bool = True,
    languages: Optional[list[str]]= None,
) -> Dict[str,Any]:

    """
    Evaluate any embedding model on MTEB STSBenchmark and return key metrics.
    Works with model names supproted by mteb,get_model()

    Returns a dict with main metrics. + raw rsults object.
    """

    if languages is None:
      languages = ["eng"]


    # 1) Load model via MTEB
    model = mteb.get_model(model_name)

    # 2) Get task objects
    tasks = mteb.get_tasks(tasks=["STSBenchmark"], languages=languages)

    # 3) Run evaluation
    res = mteb.evaluate(
        model,
        tasks=tasks,
        encode_kwargs={
        "batch_size" : batch_size,
        "normalize_embeddings" : normalize_embeddings,
        },
    )


    # 4) Extract metrics Cleanly
    # res.task_results -> lsit[TaskResults]

    tr = next((t for t in res.task_results if t.task_name == "STSBenchmark"), None)
    if tr is None:
      raise RuntimeError("STSBenchmark TaskResult not found in results")
    # Scores structure: {'Test':[{...metrics...} ]}
    test_entry = tr.scores["test"][0]
    out = {
        "model_name": res.model_name,
        "model_revision": res.model_revision,
        "main_score":float(test_entry['main_score']),
        "spearman":float(test_entry['spearman']),
        "pearson":float(test_entry['pearson']),
        "cosine_spearman":float(test_entry['cosine_spearman']),
        "cosine_pearson":float(test_entry['cosine_pearson']),
        "euclidean_spearman":float(test_entry['euclidean_spearman']),
        "manhattan_spearman":float(test_entry['manhattan_spearman']),
        "languages": test_entry.get("languages"),
        "hf_subset": test_entry.get("hf_sub_set"),
        "raw":res,
    }


    # pretty print
    print("\n=== STSBenchmark (MTEB) ===")
    print(f"Model: {out['model_name']}")
    print(f"Revision: {out['model_revision']}")
    print(f"Main (Spearman): {out['main_score']:.4f}")
    print(f"Spearman: {out['spearman']:.4f}  | Pearson : {out['pearson']:.4f}")

    print(f"Cosine Spearman:{out['cosine_spearman']:.4f}  | Euclidean spearman: {out['euclidean_spearman']:.4f} | Manhanttan Spearman:{out['manhattan_spearman']:.4f}")

    return out

In [ ]:
r1 = eval_stsbenchmark("sentence-transformers/all-MiniLM-L6-v2")
r2 = eval_stsbenchmark("BAAI/bge-base-en-v1.5")
r3 = eval_stsbenchmark("intfloat/e5-base-v2")

In [ ]:
import pandas as pd

results_df = pd.DataFrame([r1, r2, r3])
display(results_df)

How to select the best embedding model:

1. Embedding Quality ---> Check the benchmark performance:
  - MTEB leaderboard
  - BEIR benchmark

2. Dimensionality
    ex.
    
      384 dim --> lightweight

      768 dim --> balanced

      1536 dim --> high balanced but heavy

  Higher dimension:
   - Better Semsntic search  representation
   - bit the more storage cost
   - more memory usdage
   - speed impact
3. Cost
  - close source (openai embedding):
    - easy , high quality, API cost.
  - Open source:
    - free but ubfra + Scaling  cost
4. Domain Suitability
  - General model
    - all miniLM
    - OpenAI embedding

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install sentence-transformers torch

In [ ]:
from datasets import load_from_disk
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

In [ ]:
# load base model

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

In [ ]:
# with inbuilt dataset
# 2. Load dataset
#dataset = load_dataset("senetence-transformers/all-nli","triplet")
#train_dataset = dataset['train'].select(range(100_000))
#eval_dataset = dataset['dev']

In [ ]:
import zipfile
import os

zip_path = "/content/pharma_ft_data_backup.zip"     # zip file path
extract_to = "unzipped_data"     # folder were to  extract

os.makedirs(extract_to, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Unzip complete")

In [ ]:
import os

files_in_pharma_data = os.listdir('/content/pharma_ft_data')
print(files_in_pharma_data)

In [ ]:
# load your custom dataset

train_dataset = load_from_disk("/content/pharma_ft_data/train_pairs")

In [ ]:
print("dataset Loaded :", train_dataset)
print("columns : ", train_dataset.column_names)
print("shape : ", train_dataset.shape)
print("features : ", train_dataset.features)

In [ ]:
# define Loss (perfect for anchor-postive pairs)

loss = MultipleNegativesRankingLoss(model)

In [ ]:
# Training arguments

args = SentenceTransformerTrainingArguments(
    output_dir="models/pharma-embedding-ft",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=10,
    save_strategy="epoch",
)

In [ ]:
# Trainer (no evaluator needed for pair dataset)
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

In [ ]:
# train
trainer.train()

In [ ]:
#save model

model.save_pretrained("models/pharma-embedding-ft")
print("fine-tuned model successfully.")

Inference code

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("/content/models/pharma-embedding-ft")

sentences = [
    "what is amoxcilin capsule",
]

embeddings = embedding_model.encode(sentences, normalize_embeddings=True)
print(embeddings)

In [ ]:
# !pip install -U langchain
# !pip install -U langchain_openai


In [ ]:
# %pip install -qU langchain_community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# !pip install langchain_community
# !pip install faiss-cpu
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

# Re-define embedding_model and documents for this cell's scope
# Load the SentenceTransformer model (this is now just to keep the path handy)
model_path = "/content/models/pharma-embedding-ft"

# Wrap the SentenceTransformer model in HuggingFaceEmbeddings, passing the model_path as a string
embedding_model = HuggingFaceEmbeddings(model_name=model_path)

documents = [
    "Embedding fine tuning improves retrieval in RAG",
    "LoRA reduces GPU memory usage",
    "My car needs fuel.",
    "Vector databases store embeddings",
]

# build FAISS from your document list
vectorstore = FAISS.from_texts(documents, embedding_model)

# test search
query = "what are the contraindications?"
results = vectorstore.similarity_search_with_score(query, k=3)

for i,r in enumerate(results, 1):
  print(f"\n--- Result {i} ---")
  print(r[0].page_content[:500])

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
# Load document using Pypdfloader document loader

loader = PyPDFLoader("/content/pharma_ft_data/pharma_demo.pdf")
documents = loader.load()

In [ ]:
documents

In [ ]:
#loading the data and the correspond embedding into FAISS
vectorstore = FAISS.from_documents(documents, embedding_model)


In [ ]:
vectorstore.save_local("faiss_index")

In [ ]:
results = vectorstore.similarity_search("What are the warnings and precautions?", k=1)

In [ ]:
for r in results:
  print("------")
  print(r.page_content)

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
import os
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-5.4-nano")

In [ ]:
llm.invoke("hi")

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import StringPromptTemplate

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant.
    Use the following context to answer the questions. if you dont know , say you dont know.

    context:
    {context}

    question:
    {question}

    answer:
    """
)

In [ ]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("what are the contraindications for amoxycillin 500 mg?")